In [ ]:
# ======================================================
# Notebook: 4D Black-box Optimisation (Chemical Yield)
# Inputs: (20,4) | Output: (20,)
# Goal: maximise yield
# ======================================================

import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from scipy.stats import norm

# Load data
X = np.load("/mnt/data/initial_inputs.npy")      # (20,4)
y = np.load("/mnt/data/initial_outputs.npy")     # (20,)

# Gaussian Process surrogate
kernel = ConstantKernel(1.0) * Matern(nu=2.5) + WhiteKernel()
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, normalize_y=True)
gp.fit(X, y)

# Expected Improvement
def expected_improvement(X_cand, X_sample, model, xi=0.01):
    mu, sigma = model.predict(X_cand, return_std=True)
    best = np.max(model.predict(X_sample))

    mu = mu.reshape(-1,1)
    sigma = sigma.reshape(-1,1)

    improvement = mu - best - xi
    Z = improvement / sigma
    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
    ei[sigma == 0.0] = 0.0
    return ei.ravel()

# Candidate sampling within bounds
bounds = [(X[:,i].min(), X[:,i].max()) for i in range(4)]
num_candidates = 5000
X_grid = np.column_stack([
    np.random.uniform(b[0], b[1], num_candidates) for b in bounds
])

# Acquisition
ei = expected_improvement(X_grid, X, gp)

# Select next (10,4) candidates
top_idx = np.argsort(ei)[-10:]
next_points = X_grid[top_idx]

print("Next (10,4) candidate inputs:")
print(next_points)